# Affine PZ two-jet versus interval AdaQuad benchmarks

Deterministic float64 tanh-network benchmark comparing the affine polynomial-zonotope (PZ) two-jet norm enclosure with the existing interval adaptive quadrature (AdaQuad) norm path. The PZ path uses the closed-form affine enclosures for `tanh`, `tanh'`, and `tanh''`; this notebook intentionally exposes no approximation tuning knobs for the activation enclosure.

The trace helpers run one adaptive refinement pass up to a configured maximum number of refinement steps and record the partial certified result after each step from the same active-cell cache, rather than issuing many independent API calls.


## 1. Reproducible setup


In [ ]:
from __future__ import annotations

import math, random, sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "src" / "intervalnets").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from intervalnets import Interval, IntervalTensor, PZIntegrationCell, collect_pz_diagnostics, enable_interval_eval, pz_sum_squares
from intervalnets.pz_integration import integrate_over_cell, _dorfler_marking as _pz_dorfler_marking, _evaluate_squared_contribution_cache, _interval_add, _interval_width, _split_box, _sqrt_interval_nonnegative
from intervalnets.pytorch import _box_volume, _choose_split_dim, _dorfler_marking, _hessian_is_exact_zero, _interval_tensor_is_exact_constant, _jacobian_is_exact_zero, _lp_pointwise_power_bounds_refined, _sobolev_pointwise_power_bounds_refined

enable_interval_eval("slope")
SEED = 20260722
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)
print(f"repo_root={repo_root}")


## 2. Small tanh network and `IntervalTensor` domain


In [ ]:
def make_tanh_network(input_dim: int = 2, width: int = 5) -> nn.Sequential:
    return nn.Sequential(nn.Linear(input_dim, width), nn.Tanh(), nn.Linear(width, width), nn.Tanh(), nn.Linear(width, 1)).to(dtype=torch.float64)

model = make_tanh_network()
with torch.no_grad():
    for idx, param in enumerate(model.parameters()):
        gen = torch.Generator().manual_seed(SEED + idx)
        param.copy_(0.25 * torch.randn(param.shape, generator=gen, dtype=torch.float64))

# Interval domain construction with IntervalTensor.
domain = IntervalTensor.from_bounds(torch.tensor([-0.90, -0.65], dtype=torch.float64), torch.tensor([0.80, 0.70], dtype=torch.float64))
MAX_REFINEMENT_STEPS = 4
THETA = 0.5
FORWARD_REFINE_SPLITS = 1
FORWARD_REFINE_MAX_CELLS = 256
model, domain, MAX_REFINEMENT_STEPS


## 3. Direct PZ two-jet norm calls on the affine tanh enclosure path


In [ ]:
cell = PZIntegrationCell.from_affine_box(domain)
jet = model.eval_pz_twojet(cell.domain)

y_sq = pz_sum_squares(jet.Y)
j_sq = pz_sum_squares(jet.J)
h_sq = pz_sum_squares(jet.H)

l2_integrand = y_sq
w12_integrand = y_sq + j_sq
w22_integrand = w12_integrand + h_sq

def _cell_norm_from_cached_integrand(integrand):
    integral = integrate_over_cell(integrand, cell, output="interval")
    return _sqrt_interval_nonnegative(integral)

direct_pz_norms = pd.DataFrame([
    {"quantity": "L2", "bounds": _cell_norm_from_cached_integrand(l2_integrand)},
    {"quantity": "W12", "bounds": _cell_norm_from_cached_integrand(w12_integrand)},
    {"quantity": "W22", "bounds": _cell_norm_from_cached_integrand(w22_integrand)},
])
direct_pz_norms["lower"] = direct_pz_norms["bounds"].map(lambda z: float(z.lower))
direct_pz_norms["upper"] = direct_pz_norms["bounds"].map(lambda z: float(z.upper))
direct_pz_norms["width"] = direct_pz_norms["upper"] - direct_pz_norms["lower"]
direct_pz_norms.drop(columns="bounds")


In [ ]:
# Optional polynomial-zonotope multiplication diagnostics for the direct two-jet path.
# This does not change default behavior; diagnostics are collected only inside
# collect_pz_diagnostics(...) contexts.
_pz_diag_records = []
with collect_pz_diagnostics("eval_pz_twojet") as records:
    diag_jet = model.eval_pz_twojet(cell.domain)
_pz_diag_records.extend(records)

with collect_pz_diagnostics("squared_components") as records:
    diag_y_sq = pz_sum_squares(diag_jet.Y)
    diag_j_sq = pz_sum_squares(diag_jet.J)
    diag_h_sq = pz_sum_squares(diag_jet.H)
_pz_diag_records.extend(records)

with collect_pz_diagnostics("cumulative_integrands") as records:
    diag_l2_integrand = diag_y_sq
    diag_w12_integrand = diag_y_sq + diag_j_sq
    diag_w22_integrand = diag_w12_integrand + diag_h_sq
_pz_diag_records.extend(records)

with collect_pz_diagnostics("integrate_cached_integrands") as records:
    _ = integrate_over_cell(diag_l2_integrand, cell, output="interval")
    _ = integrate_over_cell(diag_w12_integrand, cell, output="interval")
    _ = integrate_over_cell(diag_w22_integrand, cell, output="interval")
_pz_diag_records.extend(records)

pz_diag_df = pd.DataFrame(_pz_diag_records)
if pz_diag_df.empty:
    pz_diag_summary = pd.DataFrame(columns=["multiplications", "raw_pair_count", "output_term_count", "max_output_degree"])
else:
    pz_diag_summary = (
        pz_diag_df.groupby("phase", dropna=False)
        .agg(
            multiplications=("raw_pair_count", "size"),
            raw_pair_count=("raw_pair_count", "sum"),
            output_term_count=("output_term_count", "sum"),
            max_output_degree=("max_output_degree", "max"),
        )
        .reset_index()
    )
pz_diag_summary


## Monomial growth diagnostics

Statically summarize polynomial-zonotope term growth for the propagated two-jet and norm integrands.


In [ ]:
from collections import Counter


def _pz_scalar_entries_for_diagnostics(z):
    if z.shape == ():
        yield z
        return
    if isinstance(z.center, torch.Tensor):
        for flat_idx in range(z.center.numel()):
            multi = tuple(int(i) for i in torch.unravel_index(torch.tensor(flat_idx, device=z.center.device), z.center.shape))
            yield z[multi]
        return

    def _fallback_scalar_indices(value, prefix=()):
        if isinstance(value, tuple):
            for idx, item in enumerate(value):
                yield from _fallback_scalar_indices(item, prefix + (idx,))
        else:
            yield prefix

    for index in _fallback_scalar_indices(z.center):
        yield z[index]


def summarize_pz_monomials(component, z):
    scalar_term_counts = [len(entry.terms) for entry in _pz_scalar_entries_for_diagnostics(z)]
    kind_counts = Counter(z.noise_kinds)
    row = {
        "component": component,
        "shape": tuple(z.shape),
        "num_noise": z.num_noise,
        "num_terms": len(z.terms),
        "max_degree": max((sum(exp) for exp in z.terms), default=0),
        "scalar_entries": len(scalar_term_counts),
        "mean_terms_per_scalar": float(np.mean(scalar_term_counts)) if scalar_term_counts else 0.0,
        "max_terms_per_scalar": max(scalar_term_counts, default=0),
        "square_pair_work_estimate": sum(term_count ** 2 for term_count in scalar_term_counts),
    }
    row.update({f"noise_kind:{kind}": count for kind, count in sorted(kind_counts.items())})
    return row


monomial_diagnostics = pd.DataFrame([
    summarize_pz_monomials("jet.Y", jet.Y),
    summarize_pz_monomials("jet.J", jet.J),
    summarize_pz_monomials("jet.H", jet.H),
    summarize_pz_monomials("l2_integrand", l2_integrand),
    summarize_pz_monomials("w12_integrand", w12_integrand),
    summarize_pz_monomials("w22_integrand", w22_integrand),
]).fillna(0)
monomial_diagnostics


## 4. Existing interval AdaQuad norm calls for comparison


In [ ]:
def norm_call(method: str, quantity: str, refinement_steps: int):
    common = dict(iterations=refinement_steps, theta=THETA)
    if method == "interval":
        common.update(forward_refine_splits=FORWARD_REFINE_SPLITS, forward_refine_max_cells=FORWARD_REFINE_MAX_CELLS)
        if quantity == "L2":
            return model.lpnorm(domain, 2.0, method="interval", **common)
        return model.sobolev_norm(domain, 2.0, order=1 if quantity == "W12" else 2, method="interval", **common)
    if quantity == "L2":
        return model.pz_l2norm(domain, p=2.0, **common)
    return model.pz_sobolev_norm(domain, p=2.0, order=1 if quantity == "W12" else 2, **common)

api_rows = []
for quantity in ["L2", "W12", "W22"]:
    for method in ["interval", "pz"]:
        t0 = time.perf_counter(); bounds = norm_call(method, quantity, MAX_REFINEMENT_STEPS)
        api_rows.append({"quantity": quantity, "method": method, "refinement_steps": MAX_REFINEMENT_STEPS, "lower": float(bounds.lower), "upper": float(bounds.upper), "width": float(bounds.upper) - float(bounds.lower), "seconds": time.perf_counter() - t0})
api_comparison = pd.DataFrame(api_rows)
api_comparison


## 5. Trace helpers for tables and convergence diagnostics


In [ ]:
def width(bounds: Interval) -> float:
    return float(bounds.upper) - float(bounds.lower)

def make_row(quantity, method, iteration, bounds, cells, elapsed, extra=None):
    return {"quantity": quantity, "method": method, "iteration": iteration, "lower": float(bounds.lower), "upper": float(bounds.upper), "width": width(bounds), "runtime_s": elapsed, "cells": cells, **(extra or {})}

def interval_power_bounds(box, quantity):
    if quantity == "L2":
        return _lp_pointwise_power_bounds_refined(model, box, 2.0, forward_refine_splits=FORWARD_REFINE_SPLITS, forward_refine_max_cells=FORWARD_REFINE_MAX_CELLS)
    return _sobolev_pointwise_power_bounds_refined(model, box, 2.0, order=1 if quantity == "W12" else 2, forward_refine_splits=FORWARD_REFINE_SPLITS, forward_refine_max_cells=FORWARD_REFINE_MAX_CELLS)

def interval_aggregate(boxes, quantity):
    total = Interval.point(0.0)
    for box in boxes:
        total = total + interval_power_bounds(box, quantity) * _box_volume(box)
    return _sqrt_interval_nonnegative(total)

def interval_indicator_split(box, quantity):
    pointwise = interval_power_bounds(box, quantity)
    indicator = _interval_width(pointwise) * _box_volume(box)
    jac = model.eval_jacobian(box)
    if quantity != "L2":
        hess = model.eval_hessian(box) if quantity == "W22" else None
        if _interval_tensor_is_exact_constant(model.eval(box)) and _jacobian_is_exact_zero(jac) and (hess is None or _hessian_is_exact_zero(hess)):
            indicator = 0.0
    return indicator, _choose_split_dim(box, jac)

def run_interval_trace(quantity, max_refinement_steps):
    boxes, rows, elapsed = [domain], [], 0.0
    for iteration in range(max_refinement_steps + 1):
        t0 = time.perf_counter(); bounds = interval_aggregate(boxes, quantity); elapsed += time.perf_counter() - t0
        rows.append(make_row(quantity, "interval AdaQuad", iteration, bounds, len(boxes), elapsed, {"refined_cells": None}))
        if iteration == max_refinement_steps: break
        t0 = time.perf_counter(); indicators, split_dims = zip(*(interval_indicator_split(box, quantity) for box in boxes)); marked = set(_dorfler_marking(list(indicators), THETA))
        boxes = [child for idx, box in enumerate(boxes) for child in (_split_box(box, split_dims[idx]) if idx in marked else (box,))]
        rows[-1]["refined_cells"] = len(marked); elapsed += time.perf_counter() - t0
    return rows

def pz_integrand_kind(quantity):
    return {"L2": "l2", "W12": "w12", "W22": "w22"}[quantity]

def pz_aggregate(cells):
    total = Interval.point(0.0)
    for cached in cells:
        total = _interval_add(total, cached.contribution)
    return _sqrt_interval_nonnegative(total)

def run_pz_trace(quantity, max_refinement_steps):
    cells = [_evaluate_squared_contribution_cache(model, domain, integrand_kind=pz_integrand_kind(quantity))]
    rows, elapsed = [], 0.0
    for iteration in range(max_refinement_steps + 1):
        t0 = time.perf_counter(); bounds = pz_aggregate(cells); elapsed += time.perf_counter() - t0
        rows.append(make_row(quantity, "PZ two-jet", iteration, bounds, len(cells), elapsed, {"refined_cells": None}))
        if iteration == max_refinement_steps: break
        t0 = time.perf_counter(); indicators = [_interval_width(c.contribution) for c in cells]; marked = set(_pz_dorfler_marking(indicators, THETA)); new_cells = []
        for idx, cached in enumerate(cells):
            if idx not in marked:
                new_cells.append(cached); continue
            for child_box in _split_box(cached.box, cached.split_dim):
                new_cells.append(_evaluate_squared_contribution_cache(model, child_box, integrand_kind=pz_integrand_kind(quantity)))
        rows[-1]["refined_cells"] = len(marked); cells = new_cells; elapsed += time.perf_counter() - t0
    return rows


## 6. Benchmark tables: width, runtime, cells, and refinement steps

The following cell performs one cached adaptive run per `(method, quantity)` pair up to `MAX_REFINEMENT_STEPS`. Each row is the partial certified result after that many refinement steps from the same run.


In [ ]:
all_rows = []
for quantity in ["L2", "W12", "W22"]:
    all_rows.extend(run_interval_trace(quantity, MAX_REFINEMENT_STEPS))
    all_rows.extend(run_pz_trace(quantity, MAX_REFINEMENT_STEPS))
results = pd.DataFrame(all_rows)
results["refinement_step"] = results.pop("iteration")
results[["quantity", "method", "refinement_step", "lower", "upper", "width", "runtime_s", "cells", "refined_cells"]]


In [ ]:
final_summary = results[results["refinement_step"] == MAX_REFINEMENT_STEPS].copy()
final_summary[["quantity", "method", "lower", "upper", "width", "runtime_s", "cells"]].sort_values(["quantity", "method"])


## 7. Diagnostic plots: convergence of enclosure width versus refinement step


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, quantity in zip(axes, ["L2", "W12", "W22"]):
    subset = results[results["quantity"] == quantity]
    for method, group in subset.groupby("method"):
        ax.plot(group["refinement_step"], group["width"], marker="o", label=method)
    ax.set_title(quantity); ax.set_xlabel("refinement step"); ax.set_ylabel("certified interval width"); ax.set_yscale("log"); ax.grid(True, which="both", alpha=0.3); ax.legend()
fig.suptitle("Enclosure-width convergence under adaptive refinement")
plt.tight_layout()


## 8. Optional deterministic autograd sanity check

This Monte Carlo/autograd estimate is not a certificate; it only provides a deterministic smoke check that sampled norm estimates are consistent with the certified intervals.


In [ ]:
def autograd_pointwise_quantities(samples):
    x = samples.clone().detach().requires_grad_(True)
    y = model(x)[:, 0]
    grad = torch.autograd.grad(y.sum(), x, create_graph=True)[0]
    hess_sq = torch.zeros_like(y)
    for i in range(x.shape[1]):
        for j in range(x.shape[1]):
            hij = torch.autograd.grad(grad[:, i].sum(), x, retain_graph=True)[0][:, j]
            hess_sq = hess_sq + hij.square()
    grad_sq = grad.square().sum(dim=1)
    return {"L2": y.square().detach().numpy(), "W12": (y.square() + grad_sq).detach().numpy(), "W22": (y.square() + grad_sq + hess_sq).detach().numpy()}

generator = torch.Generator().manual_seed(SEED)
lo = torch.tensor(domain.lower, dtype=torch.float64); hi = torch.tensor(domain.upper, dtype=torch.float64)
samples = lo + (hi - lo) * torch.rand((2048, len(lo)), generator=generator, dtype=torch.float64)
volume = float(torch.prod(hi - lo)); pointwise = autograd_pointwise_quantities(samples)
sanity_rows = []
for quantity, values in pointwise.items():
    estimate = math.sqrt(max(0.0, volume * float(np.mean(values))))
    for _, certified in final_summary[final_summary["quantity"] == quantity].iterrows():
        sanity_rows.append({"quantity": quantity, "method": certified["method"], "mc_autograd_estimate": estimate, "certified_lower": certified["lower"], "certified_upper": certified["upper"], "estimate_inside_certified_interval": certified["lower"] <= estimate <= certified["upper"]})
pd.DataFrame(sanity_rows)
